In [13]:
from kafka import KafkaConsumer

server = 'localhost:9092' #same server and topic as producer
topic_name = 'green'

In [14]:
from models import Ride, ride_deserializer

In [15]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer
)

In [4]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=1, offset=0, timestamp=1773612649828, timestamp_type=0, key=None, value=Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=127, serialized_header_size=-1)

In [5]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=1, offset=1, timestamp=1773613398643, timestamp_type=0, key=None, value=Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=127, serialized_header_size=-1)

In [16]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [17]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_green
           (PULocationID, DOLocationID, passenger_count, trip_distance,
            tip_amount, total_amount, pickup_datetime, dropoff_datetime)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID, ride.passenger_count,
         ride.trip_distance, ride.tip_amount, ride.total_amount,
         pickup_dt, dropoff_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...
Inserted 2100 rows...
Inserted 2200 rows...
Inserted 2300 rows...
Inserted 2400 rows...
Inserted 2500 rows...
Inserted 2600 rows...
Inserted 2700 rows...
Inserted 2800 rows...
Inserted 2900 rows...
Inserted 3000 rows...
Inserted 3100 rows...
Inserted 3200 rows...
Inserted 3300 rows...
Inserted 3400 rows...
Inserted 3500 rows...
Inserted 3600 rows...
Inserted 3700 rows...
Inserted 3800 rows...
Inserted 3900 rows...
Inserted 4000 rows...
Inserted 4100 rows...
Inserted 4200 rows...
Inserted 4300 rows...
Inserted 4400 r

KeyboardInterrupt: 

In [ ]:
# the code above will keep listening to the stream, waiting for next value in consumer (our producer has 1000 vaues we can see all got stored)